# Controlled context × denoiser-capacity ablation

A matched 2 × 2 experiment on the existing Gaussian AR(1) market: K1/K60 history
with Large `(32, 64, 128)` or Small `(16, 32, 64)` **denoiser** channels.
The 60-day MLP history encoder, conditioning/time dimensions, v-prediction objective,
cosine schedule, 100 diffusion steps, optimizer, seeds and 15-epoch budget are unchanged.
The Small configuration changes only `model.channels`. Hidden generator state is never
used as forecasting input; the true AR coefficient 0.4 is interpretation only.

**Hypothesis:** reducing denoiser capacity may improve generalization and may change
the K60-versus-K1 gap. Improvement is not guaranteed. Primary outcomes are CRPS,
Energy Score and predictive log-mean MSE; lag slope and diffusion loss are diagnostics.
Calibration is evaluated separately. Best checkpoints are selected on validation,
so these single-seed results are development evidence, not independent significance.
The held-out test split stays untouched.

Run All performs CPU architecture checks and **two-epoch Small pilots only**, followed
by the first 64 validation dates with 256 scenarios. Existing Large models are never
trained here. Full 15-epoch continuation/evaluation/comparison commands are Markdown.

## Setup

Enable Kaggle Internet and preferably a GPU. The established sync cell selects
`feature/synthetic-market`. All new run and inference paths use `synthetic_capacity/`.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/synthetic-market"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev]'
assert _exit_code == 0, "Dependency installation failed."

## Tests

These include actual four-arm CPU forward/loss/backward/sampling checks and tiny
checkpoint fixtures. They do not train research-sized full-budget models.

In [ ]:
!pytest -q tests/unit/test_capacity_ablation.py tests/unit/test_synthetic_diffusion_comparison.py tests/unit/test_diffusion_training.py
assert _exit_code == 0, "Capacity regression tests failed."

## Historical artifacts and prerequisites

Set these paths to restored artifacts from your own Kaggle dataset/version if available.
Large runs need both saved forecasts and the original `last.pt` or `history.csv` for
hash-verified loss curves and selected/final gaps. Missing Large artifacts do not block
the Small execution pilot; they **do** block the primary four-arm study.
No Large checkpoint is retrained, restored from a different source or manufactured.

Set `BASELINE_DIR` to compatible full mean-reverting forecasts (256 scenarios). If left
`None`, the pilot first checks the familiar saved location and otherwise generates only
64 matched baseline targets in a separate directory. Existing baseline files are reused.

In [ ]:
from pathlib import Path
import torch

LARGE_FORECAST_DIRS = {
    "K1-Large": Path("/kaggle/working/synthetic_context/k1_full_inference"),
    "K60-Large": Path("/kaggle/working/synthetic_context/k60_full_inference"),
}
LARGE_RUN_DIRS = {
    "K1-Large": Path("/kaggle/working/synthetic_context/k1"),
    "K60-Large": Path("/kaggle/working/synthetic_context/k60"),
}
BASELINE_DIR = None  # Or Path("/your/restored/full_baselines")
SMALL_ROOT = Path("/kaggle/working/synthetic_capacity")

for label, directory in LARGE_FORECAST_DIRS.items():
    missing = [name for name in ("forecast_metadata.json", "validation_scenarios.npz", "config.yaml", "standardizer.npz", "data_identity.json", "experiment.json") if not (directory / name).is_file()]
    if missing:
        print(label, "missing forecast prerequisites:", directory, missing)
    if not any((LARGE_RUN_DIRS[label] / name).is_file() for name in ("last.pt", "history.csv")):
        print(label, "missing original training history:", LARGE_RUN_DIRS[label])
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Verify configurations and architecture on CPU

All four models keep the same nominal history encoder. Its initialized weights should
match under the common seed because it is constructed before the denoiser. Different
channel widths have different denoiser tensor shapes; their weights are not claimed to
be identical. No optimizer step is taken in this check.

In [ ]:
from dataclasses import replace
import pandas as pd
from diffusion_portfolio.config import load_config
from diffusion_portfolio.models.diffusion.build import build_diffusion
from diffusion_portfolio.models.encoders.history_mask import history_mask_policy
from diffusion_portfolio.evaluation.capacity_ablation import capacity_parameter_counts

large = load_config("configs/ablations/synthetic_ar1_context.yaml")
small = load_config("configs/ablations/synthetic_ar1_capacity_small.yaml")
assert small == replace(large, model=replace(large.model, channels=(16, 32, 64)))
counts = []
initial_history = None
for size, cfg in (("Large", large), ("Small", small)):
    for keep in (1, 60):
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(cfg.seed)
            model = build_diffusion(cfg, n_assets=cfg.data.source_options["n_assets"], history_mask=history_mask_policy(cfg, keep))
            if initial_history is None:
                initial_history = {key: value.clone() for key, value in model.history_encoder.state_dict().items()}
            assert all(torch.equal(value, initial_history[key]) for key, value in model.history_encoder.state_dict().items())
            history = torch.randn(2, cfg.data.lookback, model.n_assets)
            target = torch.randn(2, 1, model.n_assets)
            output = model.training_loss(history, target)
            output.loss.backward()
            samples = model.sample(history, n_scenarios=2)
            assert output.prediction.shape == (2, model.n_assets)
            assert samples.shape == (2, 2, model.n_assets)
            assert torch.isfinite(output.loss) and torch.isfinite(samples).all()
            assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        counts.append({"arm": f"K{keep}-{size}", **capacity_parameter_counts(cfg, model.n_assets), "smoke_loss": float(output.loss.detach())})
display(pd.DataFrame(counts))

## Two-epoch pilot — K1-Small

`--stop-after-epoch 2` is an absolute boundary inside the immutable 15-epoch budget.
The shared trainer emits one flushed summary per completed epoch. The directory is
separate from all historical K1/Large artifacts.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/ablations/synthetic_ar1_capacity_small.yaml --history-keep-last 1 \
    --output-dir /kaggle/working/synthetic_capacity/k1_small --device auto --resume --stop-after-epoch 2
assert _exit_code == 0, "K1-Small pilot training failed."

In [ ]:
import matplotlib.pyplot as plt

K1_SMALL = SMALL_ROOT / "k1_small"
checkpoint = torch.load(K1_SMALL / "last.pt", map_location="cpu", weights_only=False)
history = pd.DataFrame(checkpoint["history"])
display(history)
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="K1-Small — current progress")
ax.axvline(checkpoint["best_epoch"], color="gray", linestyle="--", label="Selected epoch")
ax.set(xlabel="Epoch", ylabel="Diffusion loss")
ax.legend(["Train Loss", "Validation Loss", "Selected epoch"])
ax.grid(alpha=.3)
plt.show()

## Two-epoch pilot — K60-Small

Same configuration and seed, separate mask and checkpoint directory.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/ablations/synthetic_ar1_capacity_small.yaml --history-keep-last 60 \
    --output-dir /kaggle/working/synthetic_capacity/k60_small --device auto --resume --stop-after-epoch 2
assert _exit_code == 0, "K60-Small pilot training failed."

In [ ]:
K60_SMALL = SMALL_ROOT / "k60_small"
checkpoint = torch.load(K60_SMALL / "last.pt", map_location="cpu", weights_only=False)
history = pd.DataFrame(checkpoint["history"])
display(history)
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="K60-Small — current progress")
ax.axvline(checkpoint["best_epoch"], color="gray", linestyle="--", label="Selected epoch")
ax.set(xlabel="Epoch", ylabel="Diffusion loss")
ax.legend(["Train Loss", "Validation Loss", "Selected epoch"])
ax.grid(alpha=.3)
plt.show()

## Verify saved manifests

Original channel widths, history policy, scaler, source and checkpoint hashes must
agree. A completed full run is not relabeled as a two-epoch pilot.

In [ ]:
from diffusion_portfolio.data.preparation import load_standardizer
from diffusion_portfolio.evaluation.forecast_identity import verify_synthetic_checkpoint

for directory in (K1_SMALL, K60_SMALL):
    cfg = load_config(directory / "config.yaml")
    best = torch.load(directory / "best.pt", map_location="cpu", weights_only=False)
    manifest = verify_synthetic_checkpoint(directory, cfg, load_standardizer(directory / "standardizer.npz"), best)
    assert cfg == small and manifest["training_run"]["completed_epochs"] == 2, "This section expects two-epoch Small pilots; use the manual full-study section for completed runs."
    print(directory.name, manifest["training_contract"]["history_mask"], manifest["training_run"])

## First 64 validation targets

The evaluator restores channels, original mask and training scaler automatically.
Use the same batch size 16 and configured 256 scenarios. Output directories must
be empty or absent; choose new names for repeated or full-budget inference.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/synthetic_capacity/k1_small \
    --split validation --limit 64 --batch-size 16 --device auto \
    --output-dir /kaggle/working/synthetic_capacity/k1_small_pilot_inference
assert _exit_code == 0, "K1-Small limited inference failed."

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/synthetic_capacity/k60_small \
    --split validation --limit 64 --batch-size 16 --device auto \
    --output-dir /kaggle/working/synthetic_capacity/k60_small_pilot_inference
assert _exit_code == 0, "K60-Small limited inference failed."

## Reuse matched statistical baselines

Full Stage 4E-3A archives can supply the exact first 64 targets. If none are available,
only a small pilot baseline is generated. No validation observations enter global
parameter fitting, and no source mismatch or scenario-count mismatch is tolerated.

In [ ]:
if BASELINE_DIR is None:
    previous = Path("/kaggle/working/synthetic_context/full_baselines")
    BASELINE_DIR = previous if (previous / "report.json").is_file() else SMALL_ROOT / "pilot_baselines"
    if not (BASELINE_DIR / "report.json").is_file():
        !PYTHONUNBUFFERED=1 python -u scripts/benchmark_synthetic_forecasts.py --config configs/ablations/synthetic_ar1_capacity_small.yaml --models gaussian,ar1,ewma --split validation --n-scenarios 256 --limit 64 --seed 42 --output-dir {BASELINE_DIR}
        assert _exit_code == 0, "Small matched baseline generation failed."
BASELINE_DIR = Path(BASELINE_DIR)
assert (BASELINE_DIR / "report.json").is_file(), f"Missing baseline report: {BASELINE_DIR}"
print("Using baselines:", BASELINE_DIR)

## Small-only pilot diagnostics

This explicitly excludes completed Large models. The only pilot contrast is
K60-Small minus K1-Small. It checks execution and does not answer the full capacity
question. Original run directories provide hash-bound history for loss curves.
Predictive means average **log scenarios**, and lag slopes demean per asset.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/diagnose_capacity_ablation.py \
    --pilot --baseline-dir {BASELINE_DIR} \
    --diffusion-dirs K1-Small=/kaggle/working/synthetic_capacity/k1_small_pilot_inference K60-Small=/kaggle/working/synthetic_capacity/k60_small_pilot_inference \
    --run-dirs K1-Small=/kaggle/working/synthetic_capacity/k1_small K60-Small=/kaggle/working/synthetic_capacity/k60_small \
    --block-length 20 --n-bootstrap 1000 --bootstrap-seed 42 \
    --output-dir /kaggle/working/synthetic_capacity/pilot_diagnostics
assert _exit_code == 0, "Small-only pilot diagnostics failed."

In [ ]:
import json
from IPython.display import Image

DIAGNOSTICS = SMALL_ROOT / "pilot_diagnostics"
report = json.loads((DIAGNOSTICS / "capacity_report.json").read_text())
print(report["interpretation"])
display(pd.read_csv(DIAGNOSTICS / "summary.csv"))
display(pd.read_csv(DIAGNOSTICS / "calibration.csv"))
display(pd.read_csv(DIAGNOSTICS / "contrasts.csv"))
display(Image(filename=str(DIAGNOSTICS / "loss_curves.png")))

## Continue Small runs to 15 epochs — manual commands only

After reviewing the pilots, execute these commands manually. They continue the
same checkpoints and immutable budget. Do not train either Large model again.
The preceding plot cells can be rerun at any stage to show latest completed epochs.

```bash
python -u scripts/train_diffusion.py --config configs/ablations/synthetic_ar1_capacity_small.yaml --history-keep-last 1 --output-dir /kaggle/working/synthetic_capacity/k1_small --device cuda --resume
python -u scripts/train_diffusion.py --config configs/ablations/synthetic_ar1_capacity_small.yaml --history-keep-last 60 --output-dir /kaggle/working/synthetic_capacity/k60_small --device cuda --resume
```

## Full 1,500-date validation inference — manual commands only

Run only after both Small budgets complete. New destinations preserve pilot forecasts.
Large full-validation forecasts should be reused after restoring the configured artifacts.
If a Large archive is unavailable, restore it or evaluate its original completed checkpoint
with the existing evaluator; never substitute/retrain a different model.

```bash
python -u scripts/evaluate_diffusion.py --run-dir /kaggle/working/synthetic_capacity/k1_small --split validation --batch-size 16 --device cuda --output-dir /kaggle/working/synthetic_capacity/k1_small_validation_15epoch
python -u scripts/evaluate_diffusion.py --run-dir /kaggle/working/synthetic_capacity/k60_small --split validation --batch-size 16 --device cuda --output-dir /kaggle/working/synthetic_capacity/k60_small_validation_15epoch
```

## Four-arm comparison — manual

Use your configured original Large forecast/run paths and a full baseline directory.
The following example assumes the familiar restored Kaggle paths; replace those paths
if your archives use another location. Missing/mismatched inputs fail; no implicit
intersection or training-budget adjustment is performed. A 64-date pilot baseline is
insufficient for full comparison.

```bash
python scripts/diagnose_capacity_ablation.py \
  --baseline-dir /kaggle/working/synthetic_context/full_baselines \
  --diffusion-dirs \
    K1-Large=/kaggle/working/synthetic_context/k1_full_inference \
    K60-Large=/kaggle/working/synthetic_context/k60_full_inference \
    K1-Small=/kaggle/working/synthetic_capacity/k1_small_validation_15epoch \
    K60-Small=/kaggle/working/synthetic_capacity/k60_small_validation_15epoch \
  --run-dirs \
    K1-Large=/kaggle/working/synthetic_context/k1 \
    K60-Large=/kaggle/working/synthetic_context/k60 \
    K1-Small=/kaggle/working/synthetic_capacity/k1_small \
    K60-Small=/kaggle/working/synthetic_capacity/k60_small \
  --block-length 20 --n-bootstrap 1000 --bootstrap-seed 42 \
  --output-dir /kaggle/working/synthetic_capacity/full_diagnostics
```

Capacity effects are **Small minus Large**. Context effects are **K60 minus K1**.
The interaction is `(K60-Small − K1-Small) − (K60-Large − K1-Large)`; a negative
value narrows the signed K60-versus-K1 loss gap. Each series is formed per date before
one shared moving-block bootstrap (20 observations, 1,000 replications, seed 42).
Report effect sizes and intervals together. Block intervals assume approximately
stationary differences; they omit training-seed, checkpoint-selection and repeated
ensemble Monte Carlo uncertainty. Training/validation objective gaps are descriptive,
as training and validation use different examples/corruption. Smaller diffusion loss
or a closer lag slope alone does not establish a better predictive distribution.

## Preserve artifacts

Save both Small run directories (checkpoints, scaler, config, provenance, history and
summary), the inference directories and diagnostic CSV/JSON/PNG files with the Kaggle
version. Keep the original Large and baseline artifacts separately. There is no automatic
upload, repair, deletion, or replacement of historical experiments.